In [ ]:
import sys
sys.path.append('../')

from core import LSTransferTreeBoost
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from utils import * #only needed for xgboost
import matplotlib.pyplot as plt
from baselines import *

import forest_config as c

In [ ]:
#ablation study for transfertreeboost
for d in c.d_list:
    ablation_transfer_real = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 'method',
                                   'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'val_rmse', 'val_mae', 'rmse', 'mae'])
    for seed in c.seed_list:
        for train_size in c.train_size_list:
            for target_column in c.target_columns:
            

                #data from Sweden
                data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])
                data_sweden = data_sweden[data_sweden['area_code'] == d]
                print(len(data_sweden))
                data_sweden = data_sweden.sample(1000, random_state=seed) #random sample of 1000 source instances


                #evaluate and train on latvia 
                #data from latvia target
                data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
                data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
                data_temp, data_test = train_test_split(data_latvia, test_size=0.25, random_state=seed)
                data_train, data_val = train_test_split(data_temp, test_size=0.333, random_state=seed)
                train_size_ = int(len(data_train)*train_size)
                data_train = data_train[0:train_size_]

                #"General" base dataset (to use for transfer)
                X_source_train = np.array(data_sweden[c.predictor_columns])
                y_source_train = np.array(data_sweden[target_column]) #change this to "Dgv" to use diameter as source label!

                #Specific train and test set
                X_target_train = np.array(data_train[c.predictor_columns])
                y_target_train = np.array(data_train[target_column])

                X_target_val = np.array(data_val[c.predictor_columns])
                y_target_val = np.array(data_val[target_column])

                X_target_test = np.array(data_test[c.predictor_columns])
                y_target_test = np.array(data_test[target_column])

                print(len(X_target_train), len(X_target_val), len(X_target_test))
                for config in c.param_grid_LSTransferTreeBoost:
                    v, source_tree_size, target_tree_size, k, m_0 = config

                    method = f'LSTransferTreeBoost'
                    fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                                                target_tree_size=target_tree_size, k=k, m_0=m_0)
                    fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves = False)
                    rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
                    val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
                    mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
                    val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')

                    ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, target_column, train_size_, method, v, source_tree_size, target_tree_size, k, m_0, val_rmse, val_mae, rmse, mae]
                    ablation_transfer_real.to_csv(f'results/LSTransferTreeBoost_ablation_HGV_rs_{d}.csv') #if using Dgv as source label change HGV to DGV!